# E3 · SAM 3 smoke test and building finding (no training)

Task **E3** from `Essential_task_delegation/Next_Steps_Roadmap.md` (it was S0–S1 in the research doc).

**Goals**
- **O1:** decide whether SAM 3 can do the finding for R16, the find-then-grade model (**Gate G3**).
- **O4:** measure memory and speed on a T4. L4 is the edge version (EfficientSAM3).

**What it does**
- **Part 1, smoke test:** loads SAM 3 and runs the densest validation image in fp32, fp16 and bf16. It logs seconds per image, peak GPU memory and agreement with fp32, then keeps the fastest precision that matches fp32 (mask IoU ≥ 0.95).
- **Part 2, building finding:** the text prompt "building" on the 645 validation images, in three runs:
  - the **before** image;
  - the **after** image;
  - the **before** image as a 2×2 grid of 512 px quarters. SAM 3's detector returns at most 200 objects per prompt, and the grid allows up to 800, each zoomed 2×.
- **Per run:**
  - **building IoU:** the union of SAM 3's masks against the ground-truth building mask (classes 1–3), dataset-level as in E1;
  - **count:** SAM 3's instances against the number of ground-truth building blobs;
  - **footprint recall per class:** the share of intact, damaged and destroyed ground-truth pixels that SAM 3 marks as building. For R16, the finding must find the footprints of the destroyed buildings.
- **R9 comparison (optional):** if E1's output dataset is attached, the same numbers for R9 (TTA) decide Gate G3 here. Otherwise Cell 7 says what to attach.

**Two fixes for a T4, both in the code**
1. SAM 3 pins `numpy<2`, and no numpy 1.x wheel exists for Kaggle's Python 3.13. It's installed with `--no-deps`, and Cell 1 installs the packages it actually imports.
2. SAM 3's ViT MLP layer always runs a fused **bfloat16** op, and a T4 has no bf16 tensor cores. Cell 5 swaps it for the same maths in the active precision.

**How to run on Kaggle**
1. Settings: GPU T4 ×1, internet ON.
2. Secret `HF_TOKEN` from the Hugging Face account whose access to `facebook/sam3` was approved (the AbrarAlam account, approved 18 Oct).
3. Inputs: the DisasterM3 mirror (attached automatically). Optional: `d4-r9-val-predictions` (E1's output) for the R9 comparison.
4. First run with `SMOKE_TEST = True` (8 images). Then set it to `False` and use Save & Run All. Cell 5 prints the expected run time once it has chosen a precision; plan for about 1–3 hours.
5. Outputs in `/kaggle/working/E3/`:
   - `smoke.csv`, `sam3_s1.csv` (per image and run), `E3_summary.csv`;
   - `masks/<run>/<name>.png`: 16-bit instance maps, 0 = background;
   - `config.json`, `README.md`.

**Gate G3:** if SAM 3 on the before image is at least as good as R9 at finding buildings (building IoU), R16 uses SAM 3 for the finding. Otherwise R16 uses R9's building mask.

In [1]:
# ── Cell 1: Install SAM 3 (pinned commit) and log in to Hugging Face ────────
# SAM 3 pins numpy<2, which has no wheel for Kaggle's Python 3.13, so it is installed without its
# dependency list. These are the packages it really imports at load time (checked against the source).
import importlib.util
import os
import subprocess
import sys

SAM3_COMMIT = "7f473cfd983d5eb43040ff608f66bff0d7bdcb53"     # facebookresearch/sam3 main, 14 Oct 2026

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

if importlib.util.find_spec("sam3") is None:
    pip("--no-deps", f"git+https://github.com/facebookresearch/sam3.git@{SAM3_COMMIT}")
pip("timm>=1.0.17", "ftfy==6.1.1", "regex", "iopath", "einops", "pycocotools", "psutil")
if importlib.util.find_spec("pkg_resources") is None:          # newer setuptools dropped it; SAM 3 uses it
    pip("setuptools<81")

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

token = hf_token()
if not token:
    raise RuntimeError("Add the HF_TOKEN secret: facebook/sam3 is gated and needs an approved account.")
os.environ["HF_TOKEN"] = token        # hf_hub_download inside SAM 3 reads it from here

import torch
print(f"✓ SAM 3 installed (commit {SAM3_COMMIT[:7]}); torch {torch.__version__}, python {sys.version.split()[0]}")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (this notebook needs a GPU)'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 4.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sam3 0.1.0 requires numpy<2,>=1.26, but you have numpy 2.1.3 which is incompatible.


✓ SAM 3 installed (commit 7f473cf); torch 2.11.0+cu128, python 3.13.15
  GPU: Tesla T4


In [2]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────
import glob
import json
import re
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/E3")

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
SMOKE_TEST = False                  # True = 8 validation images
PROMPT = "building"
CONFIDENCE = 0.5                    # SAM 3's own default score threshold
RUNS = [("pre", 1), ("post", 1), ("pre", 2)]   # (which image, grid); grid 2 = four 512 px quarters
MIN_INSTANCE_PX = 10                # ignore specks
DEVICE = "cuda"

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ E3 SAM 3{'  [SMOKE TEST]' if SMOKE_TEST else ''}: prompt '{PROMPT}', runs {RUNS}")

✓ E3 SAM 3: prompt 'building', runs [('pre', 1), ('post', 1), ('pre', 2)]


In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from the D4 notebooks) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: Validation split (identical to every D4 run), before-image paths and event names ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
if SMOKE_TEST:
    val_pairs = val_pairs[:8]

# Cell 3 keeps only the after-image path; the before-image comes from the same manifest entries
pre_by_name = {Path(e["post_image_path"].replace("\\", "/")).name: e.get("pre_image_path") for e in bda_entries}
for p in val_pairs:
    p["pre_path"] = resolve_path(pre_by_name.get(Path(p["image_path"]).name))

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

events = [event_and_source(p["image_path"]) for p in val_pairs]
n_xbd = sum(1 for _, source in events if source == "xBD")
print(f"✓ Validation images: {len(val_pairs)} ({n_xbd} xBD, {len(val_pairs) - n_xbd} other); "
      f"expected 645 (560 xBD, 85 other) when SMOKE_TEST is False")
print(f"  Before-images found: {sum(p['pre_path'] is not None for p in val_pairs)} of {len(val_pairs)}")

✓ Validation images: 645 (560 xBD, 85 other); expected 645 (560 xBD, 85 other) when SMOKE_TEST is False
  Before-images found: 645 of 645


## Part 1: smoke test (memory and speed on a T4)
SAM 3 runs on the densest validation image (most ground-truth building pixels) in three precisions: three timed runs after one warm-up. The fastest precision whose union mask matches fp32 (IoU ≥ 0.95) is used in Part 2. If none runs, the cell stops with a message; record that and try 2×T4, or EfficientSAM3 (L4).

In [5]:
# ── Cell 5: Load SAM 3, then Part 1: memory and speed on a T4 ─────────────
import pandas as pd
import torch.nn.functional as F
from PIL import Image
import sam3.model.vitdet as vitdet
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.model_builder import build_sam3_image_model

def _addmm_act_any_precision(activation, linear, mat1):
    """Replaces SAM 3's fused ViT-MLP op, which always computes in bfloat16 (no bf16 tensor cores on a T4).
    Same maths, activation(linear(x)), in whatever precision the surrounding autocast uses."""
    y = F.linear(mat1, linear.weight, linear.bias)
    return F.gelu(y) if activation in (F.gelu, torch.nn.GELU) else F.relu(y)

vitdet.addmm_act = _addmm_act_any_precision        # vitdet.Mlp looks the name up at call time

t0 = time.time()
model = build_sam3_image_model(device=DEVICE)       # downloads facebook/sam3 (gated) with HF_TOKEN
processor = Sam3Processor(model, device=DEVICE, confidence_threshold=CONFIDENCE)
load_s = time.time() - t0
print(f"✓ SAM 3 loaded in {load_s:.0f} s ({sum(p.numel() for p in model.parameters()) / 1e6:.0f} M parameters)")

PRECISIONS = {"fp32": nullcontext,
              "fp16": lambda: torch.autocast("cuda", dtype=torch.float16),
              "bf16": lambda: torch.autocast("cuda", dtype=torch.bfloat16)}

def segment(image, grid=1, precision="fp32"):
    """Text prompt on an RGB uint8 image -> 16-bit instance-id map (0 = background).
    grid 2 runs the four quarters separately, so each can return up to 200 objects."""
    h, w = image.shape[:2]
    ids = np.zeros((h, w), np.uint16)
    next_id = 1
    for gy in range(grid):
        for gx in range(grid):
            y0, y1, x0, x1 = gy * h // grid, (gy + 1) * h // grid, gx * w // grid, (gx + 1) * w // grid
            with PRECISIONS[precision]():
                state = processor.set_image(Image.fromarray(image[y0:y1, x0:x1]))
                out = processor.set_text_prompt(prompt=PROMPT, state=state)
            masks = out["masks"][:, 0].cpu().numpy()
            scores = out["scores"].float().cpu().numpy()
            tile = ids[y0:y1, x0:x1]
            for k in np.argsort(scores):          # lowest score first, so higher scores win any overlap
                if masks[k].sum() >= MIN_INSTANCE_PX:
                    tile[masks[k]] = next_id
                    next_id += 1
    # ponytail: a building cut by a quarter border counts twice at grid 2; overlapping tiles + merge if it matters
    return ids

def n_instances(ids):
    return len(np.unique(ids)) - 1                 # ids painted over by a better one no longer count

# Part 1 on the densest validation image
gt_building_px = [int((cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE) > 0).sum()) for p in val_pairs]
probe = val_pairs[int(np.argmax(gt_building_px))]
probe_image = cv2.cvtColor(cv2.imread(probe["image_path"]), cv2.COLOR_BGR2RGB)
smoke_rows, ref = [], None
for precision in PRECISIONS:
    row = {"precision": precision, "sec_per_image": np.nan, "peak_gpu_gb": np.nan, "instances": np.nan,
           "iou_vs_fp32": np.nan, "error": ""}
    try:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        segment(probe_image, precision=precision)                 # warm-up
        times = []
        for _ in range(3):
            torch.cuda.synchronize()
            t = time.time()
            ids = segment(probe_image, precision=precision)
            torch.cuda.synchronize()
            times.append(time.time() - t)
        union = ids > 0
        ref = union if ref is None else ref                       # fp32 runs first
        row.update(sec_per_image=round(float(np.median(times)), 3),
                   peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2), instances=n_instances(ids),
                   iou_vs_fp32=round(float((union & ref).sum() / max((union | ref).sum(), 1)), 4))
    except Exception as e:                                        # out of memory, or a dtype the GPU can't run
        row["error"] = f"{type(e).__name__}: {str(e)[:150]}"
        torch.cuda.empty_cache()
    smoke_rows.append(row)
smoke = pd.DataFrame(smoke_rows)
smoke.to_csv(OUT_DIR / "smoke.csv", index=False)
print(f"Probe: {Path(probe['image_path']).name} ({max(gt_building_px):,} ground-truth building pixels)")
print(smoke.to_string(index=False))

usable = smoke[(smoke["error"] == "") & (smoke["iou_vs_fp32"] >= 0.95)]
if usable.empty:
    raise RuntimeError("SAM 3 ran in no precision on this GPU: record smoke.csv, then try 2×T4 or EfficientSAM3 (L4).")
PRECISION = usable.sort_values("sec_per_image")["precision"].iloc[0]
passes = sum(grid * grid for _, grid in RUNS) * len(val_pairs)
sec = float(usable.set_index("precision").loc[PRECISION, "sec_per_image"])
print(f"\n✓ Part 2 uses {PRECISION}: {passes:,} forward passes ≈ {passes * sec / 3600:.1f} h")

/usr/local/lib/python3.13/dist-packages/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.13/dist-packages/sam3/model/model_misc.py:71: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

sam3.pt: reconstructing file:   0%|          |  0.00B / 3.45GB            

sam3.pt: downloading bytes:           |  0.00B            

✓ SAM 3 loaded in 38 s (841 M parameters)
Probe: turkey_earthquake0_post_456.png (741,855 ground-truth building pixels)
precision  sec_per_image  peak_gpu_gb  instances  iou_vs_fp32 error
     fp32          1.730         4.30         11       1.0000      
     fp16          0.485         4.07         11       0.9998      
     bf16          2.859         7.65         11       0.9987      

✓ Part 2 uses fp16: 3,870 forward passes ≈ 0.5 h


## Part 2: building finding on the 645 validation images
For every run (before, after, before as a 2×2 grid) and every image:
- **building IoU:** pixels that are building in both SAM 3's union mask and the ground truth (classes 1–3), over building in either. Summed over all images, as E1's dataset-level building IoU is.
- **instances against ground-truth building blobs** (8-connected).
- **footprint recall:** the share of intact, damaged and destroyed ground-truth pixels inside SAM 3's union mask.

In [6]:
# ── Cell 6: Part 2: find buildings in the validation images (before and after) ──
from tqdm.auto import tqdm

CLASSES = ((1, "intact"), (2, "damaged"), (3, "destroyed"))

def building_blobs(mask):
    return cv2.connectedComponents((mask > 0).astype(np.uint8), connectivity=8)[0] - 1

def score_row(found, gt, instances):
    """found: bool building mask; gt: 4-class mask of the same size."""
    return {"inter": int((found & (gt > 0)).sum()), "union": int((found | (gt > 0)).sum()),
            "instances": instances, "gt_blobs": building_blobs(gt),
            **{f"gt_{c}_px": int((gt == i).sum()) for i, c in CLASSES},
            **{f"found_{c}_px": int((found & (gt == i)).sum()) for i, c in CLASSES}}

rows = []
for kind, grid in RUNS:
    run = f"{kind}_grid{grid}"
    (OUT_DIR / "masks" / run).mkdir(parents=True, exist_ok=True)
    for pair, (event, source) in tqdm(list(zip(val_pairs, events)), desc=run):
        path = pair["image_path"] if kind == "post" else pair["pre_path"]
        image = cv2.imread(path) if path else None
        gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
        if image is None or gt is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if gt.shape != image.shape[:2]:              # same guard as the D4 dataset class
            gt = cv2.resize(gt, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
        t = time.time()
        ids = segment(image, grid, PRECISION)
        sec = time.time() - t
        stem = Path(pair["image_path"]).stem
        cv2.imwrite(str(OUT_DIR / "masks" / run / f"{stem}.png"), ids)
        rows.append({"run": run, "image": f"{stem}.png", "event": event, "source": source, "sec": round(sec, 2),
                     **score_row(ids > 0, gt, n_instances(ids))})

per_image = pd.DataFrame(rows)
per_image.to_csv(OUT_DIR / "sam3_s1.csv", index=False)
print(f"✓ {len(per_image)} image runs scored ({per_image.groupby('run').size().to_dict()})")

pre_grid1:   0%|          | 0/645 [00:00<?, ?it/s]

post_grid1:   0%|          | 0/645 [00:00<?, ?it/s]

pre_grid2:   0%|          | 0/645 [00:00<?, ?it/s]

✓ 1935 image runs scored ({'post_grid1': 645, 'pre_grid1': 645, 'pre_grid2': 645})


In [ ]:
# ── Cell 7: Summary, comparison with R9 (E1's output), Gate G3 and the output files ──
# R9 (TTA) on the same images, if d4-r9-val-predictions is attached: its building mask is prediction > 0
r9_dirs = sorted(glob.glob("/kaggle/input/**/R9_tta", recursive=True))
if r9_dirs:
    r9_rows = []
    for pair, (event, source) in zip(val_pairs, events):
        name = Path(pair["image_path"]).stem + ".png"
        pred = cv2.imread(str(Path(r9_dirs[0]) / name), cv2.IMREAD_GRAYSCALE)
        gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
        if pred is not None and gt is not None and pred.shape == gt.shape:
            r9_rows.append({"run": "R9_tta (E1)", "image": name, "event": event, "source": source, "sec": np.nan,
                            **score_row(pred > 0, gt, building_blobs(pred))})
    per_image = pd.concat([per_image, pd.DataFrame(r9_rows)], ignore_index=True)

def summarize(df):
    return {"images": len(df), "building_iou": round(df["inter"].sum() / max(df["union"].sum(), 1), 4),
            "count_ratio": round(df["instances"].sum() / max(df["gt_blobs"].sum(), 1), 3),
            "count_mae": round(float((df["instances"] - df["gt_blobs"]).abs().mean()), 1),
            **{f"recall_{c}": round(df[f"found_{c}_px"].sum() / max(df[f"gt_{c}_px"].sum(), 1), 4) for _, c in CLASSES},
            "sec_per_image": round(float(df["sec"].mean()), 2) if df["sec"].notna().any() else np.nan}

summary = pd.DataFrame([{"run": run, "source": src, **summarize(g)}
                        for run, d in per_image.groupby("run", sort=False)
                        for src, g in [("all", d), *d.groupby("source")]])
summary.to_csv(OUT_DIR / "E3_summary.csv", index=False)
print(summary.to_string(index=False))
print("\ncount_ratio = SAM 3 instances / ground-truth building blobs (touching buildings form one blob, so "
      "values above 1 are expected); recall_destroyed = destroyed footprints that the finding would hand to the grading.")

# Gate G3: the better before-image run against R9, on building IoU
overall = summary[summary["source"] == "all"].set_index("run")
pre_runs = overall[overall.index.str.startswith("pre")]
best_pre = pre_runs["building_iou"].idxmax()
if "R9_tta (E1)" in overall.index:
    r9_iou = overall.loc["R9_tta (E1)", "building_iou"]
    stage1 = "SAM 3 on the before image" if overall.loc[best_pre, "building_iou"] >= r9_iou else "R9's building mask"
    gate = (f"SAM 3 {best_pre}: building IoU {overall.loc[best_pre, 'building_iou']:.4f} "
            f"(destroyed footprint recall {overall.loc[best_pre, 'recall_destroyed']:.3f}) vs R9 TTA {r9_iou:.4f} "
            f"(recall {overall.loc['R9_tta (E1)', 'recall_destroyed']:.3f}). R16's finding = {stage1}.")
else:
    gate = (f"SAM 3 {best_pre}: building IoU {overall.loc[best_pre, 'building_iou']:.4f}. To decide, attach "
            "d4-r9-val-predictions (E1's output) and rerun this cell, or compare with E1_summary.csv (R9 tta).")
print(f"\nGate G3: {gate}")

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

config = {"task": "E3", "smoke_test": SMOKE_TEST, "sam3_commit": SAM3_COMMIT, "prompt": PROMPT,
          "confidence": CONFIDENCE, "runs": RUNS, "min_instance_px": MIN_INSTANCE_PX, "precision": PRECISION,
          "model_load_s": round(load_s), "val_images": len(val_pairs), "gate_g3": gate}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# E3 results: SAM 3 building finding", "",
    f"{len(val_pairs)} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''}, prompt '{PROMPT}', "
    f"confidence {CONFIDENCE}, precision {PRECISION}, SAM 3 commit {SAM3_COMMIT[:7]}.", "",
    "## Part 1: smoke test (densest validation image)", "", as_markdown(smoke), "",
    "## Part 2: building finding", "", as_markdown(summary), "",
    "## Gate G3", "", gate, "",
    "## Files", "",
    "- `smoke.csv`, `sam3_s1.csv` (per image and run), `E3_summary.csv`, `config.json`",
    "- `masks/<run>/<name>.png`: 16-bit instance maps (0 = background), same names as the after-images",
]), encoding="utf-8")
print("✓ Wrote E3_summary.csv, config.json and README.md")

        run source  images  building_iou  count_ratio  count_mae  recall_intact  recall_damaged  recall_destroyed  sec_per_image
  pre_grid1    all     645        0.4841        0.576       18.2         0.5163          0.5483            0.4931           0.54
  pre_grid1  other      85        0.4633        0.587       29.5         0.4769          0.5987            0.5507           0.56
  pre_grid1    xBD     560        0.5018        0.574       16.5         0.5584          0.5377            0.4495           0.54
 post_grid1    all     645        0.3512        0.518       20.2         0.4199          0.4184            0.0894           0.55
 post_grid1  other      85        0.2634        0.472       35.3         0.3198          0.3315            0.1080           0.56
 post_grid1    xBD     560        0.4328        0.529       17.9         0.5269          0.4366            0.0754           0.54
  pre_grid2    all     645        0.6304        1.065       11.0         0.6796          0.7288  